# Information Bottleneck for CIFAR-10 Patches
This notebook trains an information bottleneck that observes a **10×10 patch** from CIFAR-10 as the noisy signal and compresses it into a latent vector focused on predicting the **center pixel** RGB. The objective balances reconstruction against KL divergence to highlight the information bottleneck trade-off.


In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
from torchvision import transforms

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


## Dataset of 10×10 patches
Every CIFAR-10 sample is randomly cropped to a 10×10 patch. The surrounding patch is the noisy/uncompressed input, and the RGB center pixel is the prediction target.


In [ ]:
PATCH_SIZE = 10
BATCH_SIZE = 128
MEAN = [0.5, 0.5, 0.5]
STD = [0.5, 0.5, 0.5]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

cifar_full = torchvision.datasets.CIFAR10(
    root='data', train=True, transform=transform, download=True)

class PatchCenterDataset(Dataset):
    def __init__(self, base_dataset, patch_size=PATCH_SIZE):
        self.base_dataset = base_dataset
        self.patch_size = patch_size
        self.center = patch_size // 2

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, _ = self.base_dataset[idx]
        _, height, width = image.shape
        max_top = height - self.patch_size
        max_left = width - self.patch_size

        top = torch.randint(0, max_top + 1, (1,)).item()
        left = torch.randint(0, max_left + 1, (1,)).item()

        patch = image[:, top : top + self.patch_size, left : left + self.patch_size]
        center_pixel = patch[:, self.center, self.center]
        return patch, center_pixel

train_split = 45_000
train_indices = list(range(train_split))
val_indices = list(range(train_split, len(cifar_full)))

train_subset = Subset(cifar_full, train_indices)
val_subset = Subset(cifar_full, val_indices)

train_dataset = PatchCenterDataset(train_subset)
val_dataset = PatchCenterDataset(val_subset)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print('Train patches:', len(train_dataset))
print('Validation patches:', len(val_dataset))


## Information Bottleneck model
A simple convolutional encoder outputs mean and log-variance for the latent Gaussian; after sampling we regress to the center RGB and include a β-weighted KL penalty to control compression.


In [ ]:
LATENT_DIM = 16
BETA = 0.1

class InfoBottleneck(nn.Module):
    def __init__(self, patch_size=PATCH_SIZE, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self._flatten_dim = self._get_flatten_dim(patch_size)
        self.fc_mu = nn.Linear(self._flatten_dim, latent_dim)
        self.fc_logvar = nn.Linear(self._flatten_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
        )

    def _get_flatten_dim(self, patch_size):
        dummy = torch.zeros(1, 3, patch_size, patch_size)
        with torch.no_grad():
            return self.encoder(dummy).shape[1]

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        encoded = self.encoder(x)
        mu = self.fc_mu(encoded)
        logvar = self.fc_logvar(encoded)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

model = InfoBottleneck().to(DEVICE)


## Training and evaluation loops
We track reconstruction loss, KL, and latent variance so that the bottleneck behavior can be visualized later.


In [ ]:
def kl_divergence(mu, logvar):
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.7)

EPOCHS = 15
metrics = {
    key: [] for key in [
        'train_loss',
        'train_mse',
        'train_kl',
        'train_latent_var',
        'val_loss',
        'val_mse',
        'val_kl',
        'val_latent_var',
    ]
}

def _step(loader, training: bool):
    stats = {'loss': 0.0, 'mse': 0.0, 'kl': 0.0, 'latent_var': 0.0}
    total = 0

    context = model.train if training else model.eval
    with torch.set_grad_enabled(training):
        for patches, centers in loader:
            patches = patches.to(DEVICE)
            centers = centers.to(DEVICE)
            if training:
                optimizer.zero_grad()
            preds, mu, logvar = model(patches)
            mse = F.mse_loss(preds, centers, reduction='mean')
            kl = kl_divergence(mu, logvar).mean()
            latent_var = torch.exp(logvar).mean()
            loss = mse + BETA * kl
            if training:
                loss.backward()
                optimizer.step()

            batch = patches.size(0)
            total += batch
            stats['loss'] += loss.item() * batch
            stats['mse'] += mse.item() * batch
            stats['kl'] += kl.item() * batch
            stats['latent_var'] += latent_var.item() * batch

    return {key: stats[key] / total for key in stats}

def train_epoch():
    return _step(train_loader, training=True)

def eval_epoch():
    return _step(val_loader, training=False)

for epoch in range(1, EPOCHS + 1):
    train_stats = train_epoch()
    val_stats = eval_epoch()
    scheduler.step()

    print(
        f"Epoch {epoch:02d} | train loss {train_stats['loss']:.4f} | "
        f"val loss {val_stats['loss']:.4f} | beta {BETA}"
    )

    metrics['train_loss'].append(train_stats['loss'])
    metrics['train_mse'].append(train_stats['mse'])
    metrics['train_kl'].append(train_stats['kl'])
    metrics['train_latent_var'].append(train_stats['latent_var'])
    metrics['val_loss'].append(val_stats['loss'])
    metrics['val_mse'].append(val_stats['mse'])
    metrics['val_kl'].append(val_stats['kl'])
    metrics['val_latent_var'].append(val_stats['latent_var'])


## Visualizations and reconstructions
Plotting loss curves, latent statistics, and a handful of example patches shows how β trades off compression and center-pixel fidelity.


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(metrics['train_loss'], label='train loss')
plt.plot(metrics['val_loss'], label='val loss')
plt.xlabel('epoch')
plt.ylabel('total loss')
plt.title('Loss curves')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(metrics['train_kl'], label='train KL')
plt.plot(metrics['val_kl'], label='val KL')
plt.plot(metrics['train_latent_var'], label='train latent variance')
plt.plot(metrics['val_latent_var'], label='val latent variance')
plt.xlabel('epoch')
plt.title('Latent statistics')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

EXAMPLES = 4
examples_iter = iter(val_loader)
example_patches, example_centers = next(examples_iter)
example_patches = example_patches[:EXAMPLES]
example_centers = example_centers[:EXAMPLES]
with torch.no_grad():
    preds, _, _ = model(example_patches.to(DEVICE))
preds = preds.cpu()

mean_tensor = torch.tensor(MEAN).view(3, 1, 1)
std_tensor = torch.tensor(STD).view(3, 1, 1)

def unnormalize(img):
    return torch.clamp(img * std_tensor + mean_tensor, 0, 1)

fig, axes = plt.subplots(EXAMPLES, 3, figsize=(9, EXAMPLES * 3))
if EXAMPLES == 1:
    axes = axes.reshape(1, -1)

for idx in range(EXAMPLES):
    patch = unnormalize(example_patches[idx]).permute(1, 2, 0).numpy()
    true_rgb = unnormalize(example_centers[idx]).numpy()
    pred_rgb = unnormalize(preds[idx]).numpy()

    axes[idx, 0].imshow(patch)
    axes[idx, 0].set_title(f'Patch #{idx + 1}')
    axes[idx, 0].axis('off')

    for col_idx, (rgb, title) in enumerate(zip([true_rgb, pred_rgb], ['true center', 'predicted center']), 1):
        color_patch = np.ones((40, 40, 3)) * rgb.reshape(1, 1, 3)
        axes[idx, col_idx].imshow(color_patch)
        axes[idx, col_idx].set_title(title)
        axes[idx, col_idx].axis('off')

fig.suptitle('Center pixel prediction vs ground truth', y=0.92)
plt.tight_layout()
plt.show()


## Takeaways
The center-pixel reconstruction term keeps the focus on the relevant signal while the KL penalty maintains compression, so the bottleneck encodes only the information needed for that variable.
